In [4]:
from pymongo import MongoClient
import pandas as pd
import re, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

1. Load Raw Data

Pull the scraped data from MongoDB:

In [2]:
client = MongoClient("mongodb://localhost:27017/")
db = client["Medical_Diagnosis"]
collection = db["Illnesses"]

df = pd.DataFrame(list(collection.find({}, {"_id": 0, "illness_name": 1, "sections": 1})))


2. Extract Relevant Sections

Each document has a sections list. We’ll extract key ones:

In [3]:
def extract_sections(sections):
    data = {"symptoms": "", "causes": "", "warnings": "", "recommendations": ""}
    for s in sections:
        title = s["section_title"].lower()
        text = s["description"]
        if "symptom" in title:
            data["symptoms"] += " " + text
        elif "cause" in title:
            data["causes"] += " " + text
        elif "emergency" in title or "seek" in title or "warning" in title:
            data["warnings"] += " " + text
        elif "treat" in title or "self-care" in title or "manage" in title:
            data["recommendations"] += " " + text
    return data

df = df.assign(**df["sections"].apply(lambda x: extract_sections(x)).apply(pd.Series))
df.drop(columns=["sections"], inplace=True)


3. Text Cleaning & Normalization

Use regex, lowercasing, lemmatization, and stopword removal.

In [5]:
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)           # remove extra spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)    # keep only letters
    text = text.lower()
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(tokens)

for col in ["symptoms", "causes", "warnings", "recommendations"]:
    df[col] = df[col].apply(clean_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


4. Remove Empty or Redundant Records

In [6]:
df = df[df["symptoms"].str.len() > 0]
df.drop_duplicates(subset=["illness_name"], inplace=True)


5. Save Cleaned Dataset

Store the preprocessed dataset in a new collection:

In [7]:
clean_collection = db["Preprocessed_Illnesses"]
clean_collection.delete_many({})
clean_collection.insert_many(df.to_dict(orient="records"))


InsertManyResult([ObjectId('6908918b6175017c22fe40bb'), ObjectId('6908918b6175017c22fe40bc'), ObjectId('6908918b6175017c22fe40bd'), ObjectId('6908918b6175017c22fe40be'), ObjectId('6908918b6175017c22fe40bf'), ObjectId('6908918b6175017c22fe40c0'), ObjectId('6908918b6175017c22fe40c1'), ObjectId('6908918b6175017c22fe40c2'), ObjectId('6908918b6175017c22fe40c3'), ObjectId('6908918b6175017c22fe40c4'), ObjectId('6908918b6175017c22fe40c5'), ObjectId('6908918b6175017c22fe40c6'), ObjectId('6908918b6175017c22fe40c7'), ObjectId('6908918b6175017c22fe40c8'), ObjectId('6908918b6175017c22fe40c9'), ObjectId('6908918b6175017c22fe40ca'), ObjectId('6908918b6175017c22fe40cb'), ObjectId('6908918b6175017c22fe40cc'), ObjectId('6908918b6175017c22fe40cd'), ObjectId('6908918b6175017c22fe40ce'), ObjectId('6908918b6175017c22fe40cf'), ObjectId('6908918b6175017c22fe40d0'), ObjectId('6908918b6175017c22fe40d1'), ObjectId('6908918b6175017c22fe40d2'), ObjectId('6908918b6175017c22fe40d3'), ObjectId('6908918b6175017c22fe40